# Nível 1 — Análise de Operações e PLD

Este notebook implementa o tratamento dos dados, as regras determinísticas de sinalização e a análise de um cliente sinalizado utilizando um modelo de linguagem.

A solução separa os cálculos determinísticos, realizados com pandas, da interpretação qualitativa realizada pelo LLM.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
CAMINHO_DADOS = Path("../dados/dados_nivel_1.json")

with open(CAMINHO_DADOS, "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

taxa_cambio = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])

print(f"Taxa USD/BRL: {taxa_cambio}")
print(f"Quantidade inicial de registros: {len(df)}")

display(df.head())

Taxa USD/BRL: 5.4
Quantidade inicial de registros: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [3]:
df.info()

print("\nValores ausentes:")
display(df.isna().sum().to_frame("quantidade"))

print("\nIDs duplicados:")
display(
    df[df.duplicated(subset=["id"], keep=False)]
    .sort_values("id")
)

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 2.9 KB

Valores ausentes:


,quantidade
id,0
cliente_id,0
data,1
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0



IDs duplicados:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [4]:
print("Quantidade de clientes:", df["cliente_id"].nunique())
print("Quantidade de IDs únicos:", df["id"].nunique())
print("Quantidade de registros:", len(df))

print("\nMoedas:")
display(df["moeda"].value_counts())

print("\nCanais:")
display(df["canal"].value_counts())

print("\nTipos:")
display(df["tipo"].value_counts())

Quantidade de clientes: 6
Quantidade de IDs únicos: 19
Quantidade de registros: 20

Moedas:


moeda
BRL    19
USD     1
Name: count, dtype: int64


Canais:


canal
pix        9
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64


Tipos:


tipo
transferencia_enviada     11
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

## Tratamento dos dados

Foram identificados dois problemas de qualidade:

1. A operação `OP-0007` aparece duplicada com os mesmos atributos. Foi mantida apenas uma ocorrência para evitar dupla contagem e impacto indevido nas regras determinísticas.
2. A operação `OP-0017` possui data ausente. A operação foi mantida, pois ainda é válida para análises que não dependem de data, mas a ausência foi representada como `NaT` e essa operação não participa de análises que exigem agrupamento temporal.

Além disso, os valores foram normalizados para BRL utilizando a taxa de câmbio fixa fornecida no próprio arquivo.

In [5]:
df_limpo = df.copy()

# Remove duplicidades pelo identificador da operação
df_limpo = df_limpo.drop_duplicates(subset=["id"], keep="first").copy()

# Converte a coluna de data
df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")

print("Registros antes da limpeza:", len(df))
print("Registros após a limpeza:", len(df_limpo))
print("Datas ausentes após conversão:", df_limpo["data"].isna().sum())

Registros antes da limpeza: 20
Registros após a limpeza: 19
Datas ausentes após conversão: 1


In [6]:
df_limpo["valor_brl"] = df_limpo.apply(
    lambda linha: linha["valor"] * taxa_cambio
    if linha["moeda"] == "USD"
    else linha["valor"],
    axis=1
)

display(
    df_limpo[["id", "cliente_id", "valor", "moeda", "valor_brl"]]
)

,id,cliente_id,valor,moeda,valor_brl
0,OP-0001,CLI-A-1,18100,BRL,18100.0
1,OP-0002,CLI-A-1,17300,BRL,17300.0
2,OP-0003,CLI-A-1,18800,BRL,18800.0
3,OP-0004,CLI-A-1,3300,BRL,3300.0
4,OP-0005,CLI-A-2,25900,BRL,25900.0
5,OP-0006,CLI-A-2,27000,BRL,27000.0
6,OP-0007,CLI-A-3,17200,BRL,17200.0
7,OP-0008,CLI-A-3,15200,BRL,15200.0
8,OP-0009,CLI-A-3,16100,BRL,16100.0
10,OP-0010,CLI-A-4,3800,BRL,3800.0


In [7]:
volume_por_cliente = (
    df_limpo.groupby("cliente_id", as_index=False)["valor_brl"]
    .sum()
    .rename(columns={"valor_brl": "volume_total_brl"})
    .sort_values("volume_total_brl", ascending=False)
)

print("Volume total transacionado por cliente:")
display(volume_por_cliente)

Volume total transacionado por cliente:


,cliente_id,volume_total_brl
3,CLI-A-4,79500.0
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


In [8]:
operacoes_por_canal = (
    df_limpo.groupby("canal")
    .size()
    .reset_index(name="quantidade_operacoes")
    .sort_values("quantidade_operacoes", ascending=False)
)

print("Quantidade de operações por canal:")
display(operacoes_por_canal)

Quantidade de operações por canal:


,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


## Regras determinísticas

As regras abaixo são calculadas exclusivamente com pandas. O modelo de linguagem será utilizado apenas posteriormente para interpretar os casos sinalizados.

In [9]:
# Inicializa a flag como False
df_limpo["flag_fracionamento"] = False

# Considera apenas operações com data conhecida
df_com_data = df_limpo.dropna(subset=["data"]).copy()

# Agrega por cliente e data
resumo_dia = (
    df_com_data
    .groupby(["cliente_id", "data"])
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_dia_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
    .reset_index()
)

# Aplica a regra
casos_fracionamento = resumo_dia[
    (resumo_dia["quantidade_operacoes"] >= 3)
    & (resumo_dia["soma_dia_brl"] > 50000)
    & (resumo_dia["maior_operacao_brl"] < 20000)
].copy()

display(casos_fracionamento)

,cliente_id,data,quantidade_operacoes,soma_dia_brl,maior_operacao_brl
0,CLI-A-1,2026-03-09,3,54200.0,18800.0


In [10]:
for _, caso in casos_fracionamento.iterrows():
    mascara = (
        (df_limpo["cliente_id"] == caso["cliente_id"])
        & (df_limpo["data"] == caso["data"])
    )
    df_limpo.loc[mascara, "flag_fracionamento"] = True

display(
    df_limpo[df_limpo["flag_fracionamento"]][
        ["id", "cliente_id", "data", "valor_brl", "flag_fracionamento"]
    ]
)

,id,cliente_id,data,valor_brl,flag_fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True


In [11]:
validacao_regra1 = resumo_dia[
    (
        (resumo_dia["cliente_id"] == "CLI-A-1")
        & (resumo_dia["data"] == pd.Timestamp("2026-03-09"))
    )
    |
    (
        (resumo_dia["cliente_id"] == "CLI-A-3")
        & (resumo_dia["data"] == pd.Timestamp("2026-03-05"))
    )
].copy()


validacao_regra1["regra1_sinalizada"] = (
    (validacao_regra1["quantidade_operacoes"] >= 3)
    & (validacao_regra1["soma_dia_brl"] > 50000)
    & (validacao_regra1["maior_operacao_brl"] < 20000)
)


display(validacao_regra1)

,cliente_id,data,quantidade_operacoes,soma_dia_brl,maior_operacao_brl,regra1_sinalizada
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False


### Validação da Regra 1

O cliente `CLI-A-1` foi corretamente sinalizado, pois realizou 3 operações na mesma data, totalizando mais de R$ 50.000, sem que nenhuma operação isolada atingisse R$ 20.000.

O cliente `CLI-A-3` apresenta um padrão semelhante de 3 operações na mesma data, porém a soma permanece abaixo de R$ 50.000. Por isso, não é sinalizado pela regra.

Essa comparação também demonstra a importância da remoção da operação duplicada identificada durante a limpeza.

In [12]:
# Quantidade de operações e mediana por cliente
estatisticas_cliente = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_valor_brl=("valor_brl", "median")
    )
    .reset_index()
)

display(estatisticas_cliente)

,cliente_id,quantidade_operacoes,mediana_valor_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


In [13]:
# ============================================================
# REGRA 2 — VALOR ATÍPICO
# ============================================================

# A célula pode ser executada mais de uma vez.
# Por isso, removemos colunas derivadas de execuções anteriores
# antes de recalcular as estatísticas.

colunas_antigas = [
    "quantidade_operacoes",
    "mediana_valor_brl",
    "flag_valor_atipico",
    "quantidade_operacoes_x",
    "quantidade_operacoes_y",
    "mediana_valor_brl_x",
    "mediana_valor_brl_y",
]

df_limpo = df_limpo.drop(
    columns=[
        coluna
        for coluna in colunas_antigas
        if coluna in df_limpo.columns
    ],
    errors="ignore"
)


# ============================================================
# ESTATÍSTICAS POR CLIENTE
# ============================================================

estatisticas_cliente = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_valor_brl=("valor_brl", "median")
    )
    .reset_index()
)


print("Estatísticas por cliente:")
display(estatisticas_cliente)


# ============================================================
# ADICIONA AS ESTATÍSTICAS AO DATAFRAME
# ============================================================

df_limpo = df_limpo.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left",
    validate="many_to_one"
)


# ============================================================
# APLICA A REGRA DE VALOR ATÍPICO
# ============================================================
#
# Regra:
# - cliente precisa possuir pelo menos 4 operações;
# - operação precisa ser maior que 5x a mediana
#   das operações daquele cliente.
# ============================================================

df_limpo["flag_valor_atipico"] = (
    (df_limpo["quantidade_operacoes"] >= 4)
    &
    (
        df_limpo["valor_brl"]
        > 5 * df_limpo["mediana_valor_brl"]
    )
)


# ============================================================
# RESULTADO
# ============================================================

operacoes_valor_atipico = (
    df_limpo[
        df_limpo["flag_valor_atipico"]
    ][
        [
            "id",
            "cliente_id",
            "valor_brl",
            "mediana_valor_brl",
            "quantidade_operacoes",
            "flag_valor_atipico"
        ]
    ]
    .copy()
)


print("Operações com valor atípico:")
display(operacoes_valor_atipico)

Estatísticas por cliente:


,cliente_id,quantidade_operacoes,mediana_valor_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


Operações com valor atípico:


,id,cliente_id,valor_brl,mediana_valor_brl,quantidade_operacoes,flag_valor_atipico
12,OP-0013,CLI-A-4,64800.0,5450.0,4,True


## Parte B — Análise qualitativa com LLM

Para a análise qualitativa foi selecionado o cliente `CLI-A-4`, sinalizado pela Regra 2 (valor atípico).

Os cálculos de quantidade de operações, mediana, conversão cambial e comparação com o limite foram realizados previamente com pandas. O modelo de linguagem recebe esses resultados como fatos e é utilizado somente para interpretação e redação do parecer.

In [14]:
cliente_escolhido = "CLI-A-4"

operacoes_cliente = df_limpo[
    df_limpo["cliente_id"] == cliente_escolhido
].copy()

display(
    operacoes_cliente[
        [
            "id",
            "data",
            "valor",
            "moeda",
            "valor_brl",
            "canal",
            "tipo",
            "contraparte",
            "flag_valor_atipico"
        ]
    ]
)

,id,data,valor,moeda,valor_brl,canal,tipo,contraparte,flag_valor_atipico
9,OP-0010,2026-03-03,3800,BRL,3800.0,cartao,pagamento,Alfa Comercio LTDA,False
10,OP-0011,2026-03-11,5100,BRL,5100.0,boleto,pagamento,Beta Servicos ME,False
11,OP-0012,2026-03-18,5800,BRL,5800.0,pix,transferencia_enviada,Gama Distribuidora,False
12,OP-0013,2026-03-24,12000,USD,64800.0,ted,transferencia_recebida,Zeta Importacao,True


### Consulta ao modelo e validação da resposta

A resposta do modelo é validada com `Pydantic`.

Caso o modelo retorne JSON inválido ou uma estrutura incompatível com o schema
esperado, a função não interrompe a execução: o erro é identificado e uma nova
tentativa é realizada solicitando apenas a correção do formato.

Tokens e tempo de resposta também são registrados para permitir a comparação
entre as estratégias de prompt.

In [15]:
# ============================================================
# CONFIGURAÇÃO COMPLETA — PARTE B / LLM
# ============================================================

import os
import json
import time

from pathlib import Path
from json import JSONDecodeError
from typing import Literal

from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, ValidationError


# ============================================================
# CONFIGURAÇÃO DA API
# ============================================================

pasta_atual = Path.cwd()

if (pasta_atual / ".env").exists():
    CAMINHO_ENV = pasta_atual / ".env"

elif (pasta_atual.parent / ".env").exists():
    CAMINHO_ENV = pasta_atual.parent / ".env"

else:
    raise FileNotFoundError(
        "Arquivo .env não encontrado na raiz do projeto."
    )


load_dotenv(
    dotenv_path=CAMINHO_ENV,
    override=True
)

api_key = os.getenv(
    "GROQ_API_KEY",
    ""
).strip()

if not api_key:
    raise ValueError(
        "GROQ_API_KEY não encontrada no arquivo .env."
    )


client = Groq(
    api_key=api_key
)

MODELO = "openai/gpt-oss-20b"


print("Arquivo .env encontrado:", CAMINHO_ENV)
print("Chave carregada:", True)
print("Cliente Groq criado.")


# ============================================================
# SCHEMA DA RESPOSTA
# ============================================================

class ParecerPLD(BaseModel):
    nivel_risco: Literal[
        "baixo",
        "médio",
        "alto"
    ]

    tipologia_suspeita: str

    red_flags: list[str]

    justificativa: str


# ============================================================
# EXTRAÇÃO DE JSON
# ============================================================

def extrair_json(texto: str) -> str:
    """
    Remove possíveis blocos Markdown e mantém
    apenas o objeto JSON retornado pelo modelo.
    """

    texto = (texto or "").strip()

    if texto.startswith("```"):

        linhas = texto.splitlines()

        if linhas:
            linhas = linhas[1:]

        if (
            linhas
            and linhas[-1].strip() == "```"
        ):
            linhas = linhas[:-1]

        texto = "\n".join(
            linhas
        ).strip()

    inicio = texto.find("{")
    fim = texto.rfind("}")

    if (
        inicio != -1
        and fim != -1
        and fim >= inicio
    ):
        texto = texto[
            inicio: fim + 1
        ]

    return texto


# ============================================================
# VALIDAÇÃO COM PYDANTIC
# ============================================================

def validar_parecer(texto: str):
    """
    Tenta converter a resposta para JSON
    e valida sua estrutura com Pydantic.

    O erro é tratado e retornado em vez
    de interromper o notebook.
    """

    try:

        dados_resposta = json.loads(
            extrair_json(texto)
        )

        parecer = (
            ParecerPLD.model_validate(
                dados_resposta
            )
        )

        return {
            "valido": True,
            "parecer": parecer,
            "erro": None
        }

    except (
        JSONDecodeError,
        ValidationError,
        TypeError
    ) as erro:

        return {
            "valido": False,
            "parecer": None,
            "erro": str(erro)
        }


# ============================================================
# CONSULTA AO LLM
# ============================================================

def consultar_llm(
    prompt: str,
    max_tentativas: int = 2
):
    """
    Consulta o modelo e valida a resposta.

    Caso o modelo retorne uma resposta
    fora da estrutura esperada, é feita
    uma tentativa adicional solicitando
    somente a correção do formato.
    """

    prompt_atual = prompt

    tokens_entrada_total = 0
    tokens_saida_total = 0
    tokens_total = 0

    tempo_total = 0.0

    ultimo_conteudo = ""
    ultimo_erro = None


    for tentativa in range(
        1,
        max_tentativas + 1
    ):

        inicio = time.perf_counter()

        resposta = (
            client
            .chat
            .completions
            .create(
                model=MODELO,
                messages=[
                    {
                        "role": "user",
                        "content": prompt_atual
                    }
                ],
                temperature=0
            )
        )

        tempo_chamada = (
            time.perf_counter()
            - inicio
        )

        tempo_total += tempo_chamada


        uso = resposta.usage

        tokens_entrada_total += (
            uso.prompt_tokens or 0
        )

        tokens_saida_total += (
            uso.completion_tokens or 0
        )

        tokens_total += (
            uso.total_tokens or 0
        )


        conteudo = (
            resposta
            .choices[0]
            .message
            .content
            or ""
        )

        ultimo_conteudo = conteudo


        validacao = validar_parecer(
            conteudo
        )


        if validacao["valido"]:

            return {
                "valido": True,
                "conteudo": conteudo,
                "parecer": validacao[
                    "parecer"
                ],
                "tentativas": tentativa,
                "tempo_segundos": tempo_total,
                "tokens_entrada": (
                    tokens_entrada_total
                ),
                "tokens_saida": (
                    tokens_saida_total
                ),
                "tokens_total": (
                    tokens_total
                ),
                "erro": None
            }


        ultimo_erro = validacao[
            "erro"
        ]


        if tentativa < max_tentativas:

            prompt_atual = f"""
A resposta abaixo não respeitou
o formato solicitado.

CORRIJA SOMENTE O FORMATO.

Retorne exclusivamente um objeto
JSON válido com exatamente estes campos:

{{
  "nivel_risco": "baixo | médio | alto",
  "tipologia_suspeita": "texto",
  "red_flags": ["texto"],
  "justificativa": "texto"
}}

Não adicione Markdown.
Não adicione explicações antes
ou depois do JSON.

RESPOSTA INVÁLIDA:

{conteudo}
"""


    return {
        "valido": False,
        "conteudo": ultimo_conteudo,
        "parecer": None,
        "tentativas": max_tentativas,
        "tempo_segundos": tempo_total,
        "tokens_entrada": (
            tokens_entrada_total
        ),
        "tokens_saida": (
            tokens_saida_total
        ),
        "tokens_total": tokens_total,
        "erro": ultimo_erro
    }


print(
    "Configuração da Parte B carregada."
)

Arquivo .env encontrado: d:\Teste Técnico IA - Itaú\.env
Chave carregada: True
Cliente Groq criado.
Configuração da Parte B carregada.


In [16]:
# Teste local do tratamento de uma resposta malformada.
# Não faz nova chamada à API.

resposta_malformada = """
{
    "nivel_risco": "médio"
}
"""

teste_malformado = validar_parecer(resposta_malformada)

print("Resposta válida:", teste_malformado["valido"])
print("Erro tratado sem interromper a execução:")
print(teste_malformado["erro"])

Resposta válida: False
Erro tratado sem interromper a execução:
3 validation errors for ParecerPLD
tipologia_suspeita
  Field required [type=missing, input_value={'nivel_risco': 'médio'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
red_flags
  Field required [type=missing, input_value={'nivel_risco': 'médio'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
justificativa
  Field required [type=missing, input_value={'nivel_risco': 'médio'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [17]:
# ============================================================
# PREPARAÇÃO DO CONTEXTO PARA O LLM
# ============================================================

cliente_escolhido = "CLI-A-4"


# Verifica se o tratamento determinístico já foi executado
if "df_limpo" not in globals():
    raise RuntimeError(
        "df_limpo não existe. "
        "Execute primeiro as células de tratamento e regras do Nível 1."
    )


# ============================================================
# SELECIONA AS OPERAÇÕES DO CLIENTE
# ============================================================

operacoes_cliente = (
    df_limpo[
        df_limpo["cliente_id"] == cliente_escolhido
    ]
    .copy()
)


if operacoes_cliente.empty:
    raise ValueError(
        f"Nenhuma operação encontrada para {cliente_escolhido}."
    )


# ============================================================
# ESTATÍSTICAS CALCULADAS COM PANDAS
# ============================================================

quantidade_operacoes = int(
    len(operacoes_cliente)
)

mediana_valor_brl = float(
    operacoes_cliente["valor_brl"].median()
)


# ============================================================
# COLUNAS QUE SERÃO ENVIADAS AO LLM
# ============================================================

colunas_contexto = [
    "id",
    "data",
    "valor_brl",
    "canal",
    "tipo",
    "contraparte"
]


# Inclui as flags somente se já existirem no DataFrame
if "flag_fracionamento" in operacoes_cliente.columns:
    colunas_contexto.append(
        "flag_fracionamento"
    )

if "flag_valor_atipico" in operacoes_cliente.columns:
    colunas_contexto.append(
        "flag_valor_atipico"
    )


operacoes_contexto = (
    operacoes_cliente[
        colunas_contexto
    ]
    .copy()
)


# ============================================================
# TRATA A DATA PARA O JSON
# ============================================================

operacoes_contexto["data"] = (
    operacoes_contexto["data"]
    .apply(
        lambda valor:
            None
            if pd.isna(valor)
            else valor.strftime("%Y-%m-%d")
            if hasattr(valor, "strftime")
            else str(valor)
    )
)


# ============================================================
# CONVERTE PARA ESTRUTURA COMPATÍVEL COM JSON
# ============================================================

operacoes_json = json.loads(
    operacoes_contexto.to_json(
        orient="records",
        force_ascii=False
    )
)


# ============================================================
# CONTEXTO FINAL
# ============================================================

contexto_cliente = {
    "cliente_id": cliente_escolhido,
    "quantidade_operacoes": quantidade_operacoes,
    "mediana_valor_brl": mediana_valor_brl,
    "operacoes": operacoes_json
}


print("Contexto criado com sucesso:")
print(
    json.dumps(
        contexto_cliente,
        indent=2,
        ensure_ascii=False
    )
)

Contexto criado com sucesso:
{
  "cliente_id": "CLI-A-4",
  "quantidade_operacoes": 4,
  "mediana_valor_brl": 5450.0,
  "operacoes": [
    {
      "id": "OP-0010",
      "data": "2026-03-03",
      "valor_brl": 3800.0,
      "canal": "cartao",
      "tipo": "pagamento",
      "contraparte": "Alfa Comercio LTDA",
      "flag_fracionamento": false,
      "flag_valor_atipico": false
    },
    {
      "id": "OP-0011",
      "data": "2026-03-11",
      "valor_brl": 5100.0,
      "canal": "boleto",
      "tipo": "pagamento",
      "contraparte": "Beta Servicos ME",
      "flag_fracionamento": false,
      "flag_valor_atipico": false
    },
    {
      "id": "OP-0012",
      "data": "2026-03-18",
      "valor_brl": 5800.0,
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Gama Distribuidora",
      "flag_fracionamento": false,
      "flag_valor_atipico": false
    },
    {
      "id": "OP-0013",
      "data": "2026-03-24",
      "valor_brl": 64800.0,
      "c

### Prompt 1 — abordagem direta

A primeira estratégia fornece os dados já calculados e solicita diretamente o
parecer estruturado.

O modelo não recebe a responsabilidade de recalcular soma, mediana, conversão
cambial ou aplicação das regras.

In [18]:
contexto_json = json.dumps(
    contexto_cliente,
    indent=2,
    ensure_ascii=False
)

prompt_1 = f"""
Você atua como analista de Prevenção à Lavagem de Dinheiro (PLD).

Analise o cliente abaixo utilizando somente as informações fornecidas.

Os cálculos já foram realizados previamente com pandas.
Não recalcule valores, medianas, limites ou flags.

DADOS DO CLIENTE:

{contexto_json}

Produza um parecer contendo:

- nivel_risco: baixo, médio ou alto;
- tipologia_suspeita;
- red_flags;
- justificativa.

Retorne SOMENTE um objeto JSON válido no seguinte formato:

{{
  "nivel_risco": "baixo | médio | alto",
  "tipologia_suspeita": "texto",
  "red_flags": ["texto"],
  "justificativa": "texto"
}}
"""

resultado_1 = consultar_llm(prompt_1)

print("Resposta válida:", resultado_1["valido"])
print("Tentativas:", resultado_1["tentativas"])

if resultado_1["valido"]:
    print(
        json.dumps(
            resultado_1["parecer"].model_dump(),
            indent=2,
            ensure_ascii=False
        )
    )
else:
    print("Erro:", resultado_1["erro"])

print(f"\nTempo: {resultado_1['tempo_segundos']:.2f}s")
print(f"Tokens entrada: {resultado_1['tokens_entrada']}")
print(f"Tokens saída: {resultado_1['tokens_saida']}")
print(f"Tokens total: {resultado_1['tokens_total']}")

Resposta válida: True
Tentativas: 1
{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Transferência recebida de valor atípico via TED, possivelmente lavagem de dinheiro",
  "red_flags": [
    "Valor atípico em operação de transferência recebida",
    "Transferência recebida de grande valor via TED",
    "Contraparte não identificada como cliente habitual"
  ],
  "justificativa": "O cliente realizou apenas quatro operações, com medianas de R$5.450,00. Uma operação de R$64.800,00 (TED recebida) excede em muito a mediana e está marcada como valor atípico. Não há fracionamento, mas o valor isolado indica possível tentativa de ocultar origem de recursos. A natureza da operação (recebimento via TED) e a ausência de histórico de transações semelhantes reforçam a suspeita de lavagem de dinheiro."
}

Tempo: 1.04s
Tokens entrada: 621
Tokens saída: 384
Tokens total: 1005


### Prompt 2 — abordagem orientada a evidências

A segunda estratégia adiciona restrições explícitas para reduzir inferências
não sustentadas.

Além de utilizar somente os fatos fornecidos, o modelo é orientado a não
confundir comportamento atípico com comprovação de atividade ilícita e a
considerar as limitações das evidências disponíveis.

In [19]:
# ============================================================
# PROMPT 2 — ABORDAGEM ORIENTADA A EVIDÊNCIAS
# ============================================================

# Garante que o contexto do cliente já tenha sido criado
if "contexto_cliente" not in globals():
    raise RuntimeError(
        "A variável 'contexto_cliente' não existe. "
        "Execute primeiro as células anteriores que criam o contexto do cliente."
    )


# Converte o contexto para JSON
contexto_json = json.dumps(
    contexto_cliente,
    indent=2,
    ensure_ascii=False
)


# ============================================================
# PROMPT
# ============================================================

prompt_2 = f"""
Você atua como analista de apoio à triagem de Prevenção à Lavagem de Dinheiro
(PLD).

Sua tarefa é interpretar EXCLUSIVAMENTE as evidências fornecidas abaixo.

REGRAS OBRIGATÓRIAS:

1. Não faça novos cálculos.
2. Não recalcule soma, média, mediana, conversão cambial ou limites.
3. Considere como fatos os valores e flags já calculados.
4. Não invente informações sobre origem dos recursos, intenção do cliente,
   vínculo entre contrapartes ou atividade criminosa.
5. Uma operação atípica é um sinal para investigação, não prova de ilícito.
6. Não afirme ausência de evidências que não possam ser verificadas pelos dados
   fornecidos.
7. A justificativa deve citar evidências concretas presentes no contexto.
8. Use classificação de risco proporcional às evidências disponíveis.
9. Retorne somente JSON válido, sem Markdown e sem texto adicional.
10. O campo tipologia_suspeita deve descrever somente o padrão observável.
    Não use expressões como "lavagem de dinheiro", "fraude", "origem ilícita"
    ou equivalentes sem evidência explícita nos dados.

11. Quando não houver uma flag adicional, diga que não há outra flag nos dados
    fornecidos. Não conclua que não existem outros padrões de risco que não
    tenham sido avaliados.

CONTEXTO DO CLIENTE:

{contexto_json}

Retorne exatamente esta estrutura:

{{
  "nivel_risco": "baixo | médio | alto",
  "tipologia_suspeita": "texto objetivo baseado nas evidências",
  "red_flags": [
    "apenas sinais observáveis nos dados"
  ],
  "justificativa": "explique a classificação usando somente fatos presentes no contexto"
}}
"""


# ============================================================
# CONSULTA AO LLM
# ============================================================

resultado_2 = consultar_llm(prompt_2)


# ============================================================
# RESULTADO
# ============================================================

print("Resposta válida:", resultado_2["valido"])
print("Tentativas:", resultado_2["tentativas"])

if resultado_2["valido"]:

    parecer_2 = resultado_2["parecer"].model_dump()

    print(
        json.dumps(
            parecer_2,
            indent=2,
            ensure_ascii=False
        )
    )

else:

    print("Erro:", resultado_2["erro"])


# ============================================================
# MÉTRICAS
# ============================================================

print(f"\nTempo: {resultado_2['tempo_segundos']:.2f}s")
print(f"Tokens entrada: {resultado_2['tokens_entrada']}")
print(f"Tokens saída: {resultado_2['tokens_saida']}")
print(f"Tokens total: {resultado_2['tokens_total']}")

Resposta válida: True
Tentativas: 1
{
  "nivel_risco": "médio",
  "tipologia_suspeita": "valor atípico em transferência recebida via TED",
  "red_flags": [
    "valor atípico em operação OP-0013"
  ],
  "justificativa": "A operação OP-0013 apresenta valor atípico (64800 BRL) em relação à mediana de 5450 BRL, sendo a única operação com flag_valor_atipico. Não há outras flags de fracionamento ou valor atípico em outras operações. Assim, há um sinal de risco moderado, mas não há evidências adicionais de atividade ilícita."
}

Tempo: 1.20s
Tokens entrada: 849
Tokens saída: 569
Tokens total: 1418


In [20]:
# ============================================================
# COMPARAÇÃO ENTRE OS DOIS PROMPTS
# ============================================================

comparacao_prompts = pd.DataFrame(
    [
        {
            "prompt": "Prompt 1",
            "tempo_segundos": round(
                resultado_1["tempo_segundos"],
                3
            ),
            "tokens_entrada": resultado_1[
                "tokens_entrada"
            ],
            "tokens_saida": resultado_1[
                "tokens_saida"
            ],
            "tokens_total": resultado_1[
                "tokens_total"
            ],
            "tentativas": resultado_1[
                "tentativas"
            ],
            "resposta_valida": resultado_1[
                "valido"
            ],
        },
        {
            "prompt": "Prompt 2",
            "tempo_segundos": round(
                resultado_2["tempo_segundos"],
                3
            ),
            "tokens_entrada": resultado_2[
                "tokens_entrada"
            ],
            "tokens_saida": resultado_2[
                "tokens_saida"
            ],
            "tokens_total": resultado_2[
                "tokens_total"
            ],
            "tentativas": resultado_2[
                "tentativas"
            ],
            "resposta_valida": resultado_2[
                "valido"
            ],
        },
    ]
)


print("Comparação de desempenho dos prompts:")
display(comparacao_prompts)

Comparação de desempenho dos prompts:


,prompt,tempo_segundos,tokens_entrada,tokens_saida,tokens_total,tentativas,resposta_valida
0,Prompt 1,1.039,621,384,1005,1,True
1,Prompt 2,1.202,849,569,1418,1,True


### Comparação entre os prompts

As duas estratégias produziram respostas estruturadas válidas, mas chegaram a
classificações diferentes para o cliente `CLI-A-4`.

O **Prompt 1** classificou o caso como risco **alto**, utilizando 1.005 tokens
e apresentando latência de aproximadamente 1,04 s nesta execução. Apesar de
receber os valores e flags já calculados com pandas, a resposta fez inferências
mais fortes do que os dados permitem. Por exemplo, mencionou possível tentativa
de ocultar a origem dos recursos e tratou a contraparte como não habitual,
conclusões que não são diretamente comprovadas pelo contexto fornecido.

O **Prompt 2** classificou o caso como risco **médio**, utilizando 1.418 tokens
e apresentando latência de aproximadamente 1,20 s. Embora tenha consumido mais
tokens nesta execução, sua análise ficou mais aderente às evidências
disponíveis. O modelo destacou a operação `OP-0013` como valor atípico e
descreveu a tipologia de forma objetiva como uma transferência recebida via TED
com valor atípico, sem transformar automaticamente a sinalização em prova de
lavagem de dinheiro ou fraude.

Considero o **Prompt 2 mais adequado** para este cenário, pois suas instruções
mais restritivas reduziram inferências não sustentadas e mantiveram uma
separação mais clara entre os cálculos determinísticos realizados com pandas e
a interpretação qualitativa realizada pelo LLM.

A diferença de tokens e latência observada corresponde apenas a esta execução.
Como respostas de modelos de linguagem podem variar entre chamadas, esses
valores não devem ser interpretados como um benchmark definitivo de desempenho.